In [6]:
import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD & AUDIT AWAL
# ==========================================
df = pd.read_csv('../data/raw/dataset.csv')
total_awal = len(df)
print(f"Total data awal: {total_awal} baris")

# Bersihkan spasi di nama kolom
df.columns = df.columns.str.strip()

# ==========================================
# 2. SELEKSI KOLOM & HAPUS DUPLIKAT
# ==========================================
# Pilih kolom yang dibutuhkan saja
df = df[['JK', 'Usia', 'Berat', 'Tinggi', 'LiLA', 'Status Gizi']]

# Hapus data yang sama persis (Duplikat)
df = df.drop_duplicates()
print(f"Data setelah hapus duplikat: {len(df)} baris")

# ==========================================
# 3. PEMBERSIHAN FORMAT (STRING TO NUMERIC)
# ==========================================

# Fungsi konversi usia (Tahun/Bulan ke angka Bulan)
def convert_usia(x):
    x = str(x).lower()
    angka = ''.join(filter(str.isdigit, x))
    if angka == '':
        return None
    if 'tahun' in x:
        return int(angka) * 12
    return int(angka)

# Terapkan konversi usia
df['Usia'] = df['Usia'].apply(convert_usia)

# Konversi Jenis Kelamin ke angka (L=1, P=0)
df['JK'] = df['JK'].astype(str).str.upper().str.strip()
df['JK'] = df['JK'].map({'L': 1, 'P': 0, 'LAKI-LAKI': 1, 'PEREMPUAN': 0})

# Paksa kolom lain jadi angka (Numerik)
df['Berat'] = pd.to_numeric(df['Berat'], errors='coerce')
df['Tinggi'] = pd.to_numeric(df['Tinggi'], errors='coerce')
df['LiLA'] = pd.to_numeric(df['LiLA'], errors='coerce')

# Ubah angka 0 menjadi NaN supaya bisa diisi dengan median
df['LiLA'] = df['LiLA'].replace(0, np.nan)
df['Berat'] = df['Berat'].replace(0, np.nan)

# ==========================================
# 4. STRATEGI PENYELAMATAN (IMPUTASI MEDIAN)
# ==========================================
# Mengisi data kosong (NaN) dengan nilai tengah (Median) 
# agar baris data tidak perlu dihapus semua.

df['Berat'] = df['Berat'].fillna(df['Berat'].median())
df['LiLA'] = df['LiLA'].fillna(df['LiLA'].median())

# ==========================================
# 5. DROP DATA YANG TIDAK BISA DISELAMATKAN
# ==========================================
# Data yang WAJIB ada: JK, Usia, Tinggi, dan Status Gizi
# Jika ini kosong, baris tersebut tidak bisa dipakai untuk ML
df = df.dropna(subset=['JK', 'Usia', 'Tinggi', 'Status Gizi'])

# Reset nomor index agar rapi
df = df.reset_index(drop=True)

# ==========================================
# 6. SIMPAN & LAPORAN
# ==========================================
total_akhir = len(df)
df.to_csv('../data/processed/dataset_bersih.csv', index=False)

print("\n--- LAPORAN PEMBERSIHAN DATA ---")
print(f"Total data awal           : {total_awal} baris")
print(f"Total data bersih (SIAP)  : {total_akhir} baris")
print(f"Data terbuang (Incomplete): {total_awal - total_akhir} baris")
print("\nContoh data hasil penyelamatan:")
print(df.head())

Total data awal: 1363 baris
Data setelah hapus duplikat: 1017 baris

--- LAPORAN PEMBERSIHAN DATA ---
Total data awal           : 1363 baris
Total data bersih (SIAP)  : 1016 baris
Data terbuang (Incomplete): 347 baris

Contoh data hasil penyelamatan:
    JK  Usia  Berat  Tinggi  LiLA        Status Gizi
0  0.0  48.0    8.3    79.5  14.0        Gizi Kurang
1  0.0  60.0    9.7    85.6  14.0        Gizi Kurang
2  1.0  60.0   11.0    92.5  14.4        Gizi Kurang
3  1.0  36.0    5.7    57.0  14.0  Resiko Gizi Lebih
4  0.0  60.0    8.6    78.4  13.5          Gizi Baik


In [10]:
# ===============================
# PREDIKSI STUNTING RANDOM FOREST
# ===============================

# 1️⃣ Import library
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score

# -------------------------------
# 2️⃣ Load dataset
df = pd.read_csv('../data/processed/dataset_bersih.csv')

# -------------------------------
# 3️⃣ Data prep
# Pastikan tipe data benar tanpa perlu manipulasi string lagi
df['JK'] = df['JK'].astype(int)
df['Usia'] = df['Usia'].astype(float)
df['Berat'] = df['Berat'].astype(float)
df['Tinggi'] = df['Tinggi'].astype(float)
df['LiLA'] = df['LiLA'].astype(float)

# -------------------------------
# 4️⃣ Load data WHO & Hitung Z-Score
boys = pd.read_csv('../data/references/tab_lhfa_boys_p_0_5.csv')
girls = pd.read_csv('../data/references/tab_lhfa_girls_p_0_5.csv')

def get_who_row(usia, jk):
    table = boys if jk == 1 else girls
    # Pastikan usia dibulatkan agar cocok dengan kolom 'Month' di tabel WHO
    usia_int = int(round(usia))
    row = table.iloc[(table['Month'] - usia_int).abs().argsort()[:1]].iloc[0]
    return row

def hitung_zscore(tinggi, usia, jk):
    row = get_who_row(usia, jk)
    L, M, S = row['L'], row['M'], row['S']
    z = ((tinggi / M) ** L - 1) / (L * S)
    return z

df['Zscore_TB_U'] = df.apply(
    lambda x: hitung_zscore(x['Tinggi'], x['Usia'], x['JK']),
    axis=1
)

# -------------------------------
# 5️⃣ Buat label stunting
def klasifikasi_stunting(z):
    if z < -3: return 0   # Sangat Pendek (Severely Stunted)
    elif z < -2: return 1  # Pendek (Stunted)
    else: return 2        # Normal

df['Status_Stunting'] = df['Zscore_TB_U'].apply(klasifikasi_stunting)

# 6️⃣ Siapkan fitur & label
X = df[['JK', 'Usia', 'Berat', 'Tinggi', 'LiLA', ]]
y = df['Status_Stunting']

# -------------------------------
# 7️⃣ Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------------
# 8️⃣ Oversampling SMOTE untuk training set
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# -------------------------------
# 9️⃣ Train Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

# -------------------------------
# 🔟 Prediksi & evaluasi
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# -------------------------------
# 1️⃣1️⃣ Optional: Cross-validation
scores = cross_val_score(rf, X, y, cv=5)
print(f"Mean Accuracy: {scores.mean()}")


Accuracy: 0.9901960784313726

Confusion Matrix:
 [[169   0   0]
 [  1   9   0]
 [  0   1  24]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00       169
           1       0.90      0.90      0.90        10
           2       1.00      0.96      0.98        25

    accuracy                           0.99       204
   macro avg       0.96      0.95      0.96       204
weighted avg       0.99      0.99      0.99       204

Mean Accuracy: 0.982285327924273


In [11]:
import joblib

# Simpan model yang sudah dilatih (rf adalah variabel dari kode keduamu)
joblib.dump(rf, 'model_rf_stunting.pkl')
print("Model berhasil disimpan! ✅")

Model berhasil disimpan! ✅
